# Задача на табличных данных

В данной работе @Данил Растяпин и @Глеб Плаксин будут делать задачу, основанную на табличных данных 


Поставленная задача: __предсказывать риск аварии с пострадавшими на основе табличных признаков.__

Было решено взять датасет: [Cincinnati Car Crash Data](https://www.kaggle.com/datasets/steverusso/cincinnati-car-crash-data)

В качестве бизнес-заказчика можно рассматривать страховую компанию, которая занимается автострахованием и страхованием жизни. Компании важно заранее оценивать, какие ДТП с большей вероятностью приводят к травмам или смерти, так как именно такие случаи связаны с более высокими страховыми выплатами.

Мы хотим предсказывать вероятность того, что авария приведет к пострадавшим. 

Соответственно, наша задача — бинарная классификация:

- `0` — авария без пострадавших, только материальный ущерб;
- `1` — авария с пострадавшими или летальным исходом.

Для решения задачи мы будем использовать полносвязную нейронную сеть, так как данные представлены в табличном виде: каждая авария описывается набором признаков, таких как погода, освещение, тип дороги, район, тип столкновения и характеристики участников.

## Обоснование корректности применения полносвязной нейронной сети 
Небольшой дисклеймер: для решения задачи на табличных данных вполне корректно использовать полносвязную нейронную сеть, так как данные представлены в виде набора признаков, описывающих каждый объект наблюдения.

Например, x = [признак1, признак2, итд...]

Ее главное преимущество в сравнении с базовыми линейными моделями заключается в способности автоматически находить сложные нелинейные зависимости в данных, избавляя от необходимости вручную конструировать признаки. Это же __круто__!!! При всем при этом это также очень важно.



In [42]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

In [43]:
df = pd.read_csv("cincinnati_traffic_crash_data__cpd.csv").drop(columns = 'Unnamed: 0')
df.head()

/var/folders/lk/0xgrnlvn06nf17pvwxr3n6980000gn/T/ipykernel_77919/3656284645.py:1: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cincinnati_traffic_crash_data__cpd.csv").drop(columns = 'Unnamed: 0')


,ADDRESS_X,LATITUDE_X,LONGITUDE_X,AGE,COMMUNITY_COUNCIL_NEIGHBORHOOD,CPD_NEIGHBORHOOD,CRASHDATE,CRASHLOCATION,CRASHSEVERITY,CRASHSEVERITYID,...,LOCALREPORTNO,MANNEROFCRASH,ROADCONDITIONSPRIMARY,ROADCONTOUR,ROADSURFACE,SNA_NEIGHBORHOOD,TYPEOFPERSON,WEATHER,ZIP,UNITTYPE
0,63XX GRACELY,39.107808,-84.688195,31-40,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,145004877,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",SAYLER PARK,D - DRIVER,1 - CLEAR,45233.0,03 - MID SIZE
1,9XX CHATEAU AV,39.108110,-84.560280,18-25,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,...,155002081,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,O - OCCUPANT,1 - CLEAR,45204.0,02 - COMPACT
2,30XX READING RD,39.135486,-84.496520,18-25,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,155010090,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,O - OCCUPANT,1 - CLEAR,45206.0,04 - FULL SIZE
3,36XX READING RD,39.147889,-84.489222,61-70,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,185005525,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,D - DRIVER,1 - CLEAR,45229.0,07 - PICKUP
4,37XX WARSAW AV,39.110989,-84.573138,31-40,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,185012267,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,D - DRIVER,1 - CLEAR,45205.0,04 - FULL SIZE


In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 258672 entries, 0 to 258671
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ADDRESS_X                       258669 non-null  object 
 1   LATITUDE_X                      258672 non-null  float64
 2   LONGITUDE_X                     258672 non-null  float64
 3   AGE                             258672 non-null  object 
 4   COMMUNITY_COUNCIL_NEIGHBORHOOD  253006 non-null  object 
 5   CPD_NEIGHBORHOOD                252959 non-null  object 
 6   CRASHDATE                       258669 non-null  object 
 7   CRASHLOCATION                   194021 non-null  object 
 8   CRASHSEVERITY                   258672 non-null  object 
 9   CRASHSEVERITYID                 258672 non-null  float64
 10  DATECRASHREPORTED               258670 non-null  object 
 11  DAYOFWEEK                       258671 non-null  object 
 12  GENDER          

## Интерпретация полученных признаков
| №  | Признак                                      | Тип признака       | Что это |
|----|----------------------------------------------|--------------------|--------|
| 1  | Address_X         | Текстовый      | Адрес с маскированным домом |
| 2  | COMMUNITY_COUNCIL_NEIGHBORHOOD             | Категориальный      | Район |
| 3  | CPD_NEIGHBORHOOD         | Категориальный      | Ближайшее отделение полиции |
| 4  | CPD_NEIGHBORHOOD             | Категориальный      | Тип дороги, где произошла авария |
| 5  | CRASHSEVERITY                            | Категориальный    | Летальность аварии (сразу всмятку или лайтово) |
| 6  | INSTANCEID                                      | Числовой        | ID аварии|
| 7  | LIGHTCONDITIONSPRIMARY                      | Категориальный         | Небо во время аварии |
| 8  |  MANNEROFCRASH                           | Категориальный         |  Тип аварии |
| 9 | TYPEOFPERSON                                    | Категориальный     | Кто попал в аварию (пассажир, пешеход, водитель) |
| 10 |  UNITTYPE                             | Категориальный        | Какой вид автомобиля |

In [45]:
df.CRASHSEVERITY.unique()

array(['3 - PROPERTY DAMAGE ONLY (PDO)', '2 - INJURY', '1 - FATAL INJURY',
       '5 - PROPERTY DAMAGE ONLY', '4 - INJURY POSSIBLE',
       '3 - MINOR INJURY SUSPECTED', '2 - SERIOUS INJURY SUSPECTED',
       '1 - FATAL'], dtype=object)

In [46]:
df.INJURIES.unique()

array(['1 - NO INJURY / NONE REPORTED', '3 - NON-INCAPACITATING',
       '5 - NO APPARENTY INJURY', '4 - POSSIBLE INJURY',
       '3 - SUSPECTED MINOR INJURY', '2 - POSSIBLE', '4 - INCAPACITATING',
       '2 - SUSPECTED SERIOUS INJURY', '5 - FATAL', nan, '1 - FATAL'],
      dtype=object)

In [47]:
temp = df[['INJURIES', 'CRASHSEVERITY','ADDRESS_X']].groupby(['CRASHSEVERITY','INJURIES']).count()
temp

ADDRESS_X
CRASHSEVERITY                  INJURIES                                
1 - FATAL                      1 - FATAL                             66
                               1 - NO INJURY / NONE REPORTED          3
                               2 - SUSPECTED SERIOUS INJURY          26
                               3 - SUSPECTED MINOR INJURY            32
                               4 - POSSIBLE INJURY                   13
                               5 - FATAL                              2
                               5 - NO APPARENTY INJURY               45
1 - FATAL INJURY               1 - NO INJURY / NONE REPORTED        110
                               2 - POSSIBLE                          13
                               3 - NON-INCAPACITATING                42
                               4 - INCAPACITATING                    52
                               5 - FATAL                            172
2 - INJURY                     1 - NO INJURY / NONE REPORTED      21196
                               2 - POSSIBLE                       15297
                               3 - NON-INCAPACITATING             10317
                               4 - INCAPACITATING                  2007
                               4 - POSSIBLE INJURY                    4
                               5 - NO APPARENTY INJURY                2
2 - SERIOUS INJURY SUSPECTED   1 - NO INJURY / NONE REPORTED          8
                               2 - POSSIBLE                           5
                               2 - SUSPECTED SERIOUS INJURY         486
                               3 - NON-INCAPACITATING                 4
                               3 - SUSPECTED MINOR INJURY           154
                               4 - INCAPACITATING                     8
                               4 - POSSIBLE INJURY                   56
                               5 - NO APPARENTY INJURY              303
3 - MINOR INJURY SUSPECTED     1 - NO INJURY / NONE REPORTED          9
                               2 - SUSPECTED SERIOUS INJURY           1
                               3 - NON-INCAPACITATING                 8
                               3 - SUSPECTED MINOR INJURY          4954
                               4 - POSSIBLE INJURY                  653
                               5 - NO APPARENTY INJURY             3713
3 - PROPERTY DAMAGE ONLY (PDO) 1 - NO INJURY / NONE REPORTED     144522
                               2 - POSSIBLE                           1
                               5 - NO APPARENTY INJURY               16
4 - INJURY POSSIBLE            1 - NO INJURY / NONE REPORTED          2
                               2 - POSSIBLE                           3
                               4 - POSSIBLE INJURY                 4201
                               5 - NO APPARENTY INJURY             3584
5 - PROPERTY DAMAGE ONLY       1 - NO INJURY / NONE REPORTED         64
                               4 - POSSIBLE INJURY                    1
                               5 - NO APPARENTY INJURY            46293

In [48]:
df.columns = df.columns.str.lower()

In [49]:
temp = df[['crashseverity', 'crashseverityid']].groupby(['crashseverity','crashseverityid']).count()
temp

,
crashseverity,crashseverityid
1 - FATAL,201901.0
1 - FATAL INJURY,1.0
2 - INJURY,2.0
2 - SERIOUS INJURY SUSPECTED,201902.0
3 - MINOR INJURY SUSPECTED,201903.0
3 - PROPERTY DAMAGE ONLY (PDO),3.0
4 - INJURY POSSIBLE,201904.0
5 - PROPERTY DAMAGE ONLY,201905.0


Наверное, в какой-то момент изменилось, то, как кодируют степень тяжести нанесенного ущерба. Или для разных городов разные коды

In [ ]:
# Переводим в строку без '.0' (если нет пропусков) и проверяем, что длина равна 6 цифрам
df['id6figures'] = df['crashseverityid'].dropna().astype(int).astype(str).str.len() == 6

# Заполняем пропуски (если в исходной колонке были NaN) значением False
df['id6figures'] = df['id6figures'].fillna(False).astype(int)
distribution = pd.crosstab(df['crashdate'], df['id6figures'])


In [51]:
df.drop(columns = ['zip', 'address_x',], inplace = True)

In [52]:
df

,latitude_x,longitude_x,age,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,datecrashreported,...,localreportno,mannerofcrash,roadconditionsprimary,roadcontour,roadsurface,sna_neighborhood,typeofperson,weather,unittype,id6figures
0,39.107808,-84.688195,31-40,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,06/17/2014 05:29:00 PM,...,145004877,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",SAYLER PARK,D - DRIVER,1 - CLEAR,03 - MID SIZE,0
1,39.108110,-84.560280,18-25,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,02/15/2015 03:10:00 PM,...,155002081,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,O - OCCUPANT,1 - CLEAR,02 - COMPACT,0
2,39.135486,-84.496520,18-25,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,07/23/2015 11:54:00 PM,...,155010090,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,O - OCCUPANT,1 - CLEAR,04 - FULL SIZE,0
3,39.147889,-84.489222,61-70,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/21/2018 01:16:00 PM,...,185005525,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,D - DRIVER,1 - CLEAR,07 - PICKUP,0
4,39.110989,-84.573138,31-40,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,09/01/2018 07:59:00 PM,...,185012267,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258667,39.159659,-84.402688,61-70,MADISONVILLE,MADISONVILLE,04/17/2018 04:46:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/17/2018 04:51:00 PM,...,185005351,2 - REAR-END,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",MADISONVILLE,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0
258668,39.140463,-84.602730,18-25,WESTWOOD,WESTWOOD,11/06/2014 10:21:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,11/06/2014 10:22:00 PM,...,145011351,6 - ANGLE,02 - WET,1 - STRAIGHT LEVEL,1 - CONCRETE,WESTWOOD,D - DRIVER,1 - CLEAR,03 - MID SIZE,0
258669,39.113840,-84.592351,51-60,WEST PRICE HILL,WEST PRICE HILL,01/28/2015 02:41:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2015 02:41:00 PM,...,155001202,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WEST PRICE HILL,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0
258670,39.136956,-84.605497,26-30,WESTWOOD,WESTWOOD,05/31/2014 12:20:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,05/31/2014 12:26:00 PM,...,145004142,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WESTWOOD,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0


### Несколько пассажиров

Также в ходе анализа было выявленно, что localreportno иногда повторяются. 

In [53]:
df[df['localreportno'] == 145001250]

,latitude_x,longitude_x,age,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,datecrashreported,...,localreportno,mannerofcrash,roadconditionsprimary,roadcontour,roadsurface,sna_neighborhood,typeofperson,weather,unittype,id6figures
110049,39.098891,-84.533526,51-60,QUEENSGATE,QUEENSGATE,02/13/2014 01:10:00 PM,08 - OFF RAMP,2 - INJURY,2.0,02/13/2014 01:12:00 PM,...,145001250,2 - REAR-END,02 - WET,4 - CURVE GRADE,1 - CONCRETE,QUEENSGATE,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0
184132,39.099991,-84.534216,51-60,QUEENSGATE,QUEENSGATE,02/13/2014 01:10:00 PM,08 - OFF RAMP,2 - INJURY,2.0,02/13/2014 01:12:00 PM,...,145001250,2 - REAR-END,02 - WET,4 - CURVE GRADE,1 - CONCRETE,QUEENSGATE,O - OCCUPANT,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0
200745,39.100121,-84.533446,18-25,QUEENSGATE,QUEENSGATE,02/13/2014 01:10:00 PM,08 - OFF RAMP,2 - INJURY,2.0,02/13/2014 01:12:00 PM,...,145001250,2 - REAR-END,02 - WET,4 - CURVE GRADE,1 - CONCRETE,QUEENSGATE,D - DRIVER,1 - CLEAR,03 - MID SIZE,0


Например, здесь мы видим что в одной аварии участвовало три человека ———— две женщины 51-60 и мужчина 18-25, причем одна женщина и мужчина отмечены оба как "DRIVER" и они оба без травм, а вот у второй пассажирки травма "NON-INCAPACITATING", мешающая повседневной жизни. 

### Агреггируем строки аварии

Так как один localreportno может появляться несколько раз, мы можем понять, что одна строка датасета — это один из участников аварии(>=1)

Для бизнес-задачи нам нужно предсказывать тяжесть аварии целиком, поэтому дальше агрегируем данные до уровня одной аварии.

Взглянем на то, сколько строк может приходиться на одну аварию

In [55]:
df.groupby('localreportno').size().sort_values(ascending=False)

localreportno
135001864    43
185005626    33
175003760    27
175003358    26
195018224    21
             ..
145011765     1
145011763     1
145011761     1
195003901     1
165018760     1
Length: 133873, dtype: int64

In [56]:
df[df['localreportno'] == 135001864]

,latitude_x,longitude_x,age,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,datecrashreported,...,localreportno,mannerofcrash,roadconditionsprimary,roadcontour,roadsurface,sna_neighborhood,typeofperson,weather,unittype,id6figures
2927,39.180801,-84.519626,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
5651,39.182311,-84.520696,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
11093,39.180991,-84.519686,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
19320,39.181191,-84.519716,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
21218,39.181251,-84.519876,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
24425,39.181701,-84.520576,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
28016,39.181751,-84.520926,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
28080,39.181211,-84.520096,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
31243,39.181371,-84.521216,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0
34505,39.180901,-84.521086,UNDER 18,SPRING GROVE - WINTON HILLS,WINTON HILLS,01/28/2013 02:30:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2013 02:33:00 PM,...,135001864,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WINTON HILLS,O - OCCUPANT,2 - CLOUDY,"22 - BUS (16+ SEATS, INCLUDING THE DRIVER)",0


На этом примере мы видим огромный автобус с детишками на 43 человек. Вело автобус два водителя. Взглянем на распределение 

In [57]:
import plotly.express as px

In [69]:
reports_count = df.groupby('localreportno').size().reset_index(name='rows_per_accident')
px.histogram(reports_count, x='rows_per_accident', title='строк на одну аварию')

Смертельных аварий очень мало, модель может плоховато обучиться на редком классе. Поэтому берем за таргет наличие пострадавших. 

In [70]:
df['target'] = (~df['crashseverity'].str.contains('PROPERTY DAMAGE ONLY', case=False, na=False)).astype(int)
df

,latitude_x,longitude_x,age,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,datecrashreported,...,mannerofcrash,roadconditionsprimary,roadcontour,roadsurface,sna_neighborhood,typeofperson,weather,unittype,id6figures,target
0,39.107808,-84.688195,31-40,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,06/17/2014 05:29:00 PM,...,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",SAYLER PARK,D - DRIVER,1 - CLEAR,03 - MID SIZE,0,0
1,39.108110,-84.560280,18-25,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,02/15/2015 03:10:00 PM,...,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,O - OCCUPANT,1 - CLEAR,02 - COMPACT,0,1
2,39.135486,-84.496520,18-25,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,07/23/2015 11:54:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,O - OCCUPANT,1 - CLEAR,04 - FULL SIZE,0,0
3,39.147889,-84.489222,61-70,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/21/2018 01:16:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,D - DRIVER,1 - CLEAR,07 - PICKUP,0,0
4,39.110989,-84.573138,31-40,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,09/01/2018 07:59:00 PM,...,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258667,39.159659,-84.402688,61-70,MADISONVILLE,MADISONVILLE,04/17/2018 04:46:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/17/2018 04:51:00 PM,...,2 - REAR-END,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",MADISONVILLE,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0,0
258668,39.140463,-84.602730,18-25,WESTWOOD,WESTWOOD,11/06/2014 10:21:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,11/06/2014 10:22:00 PM,...,6 - ANGLE,02 - WET,1 - STRAIGHT LEVEL,1 - CONCRETE,WESTWOOD,D - DRIVER,1 - CLEAR,03 - MID SIZE,0,0
258669,39.113840,-84.592351,51-60,WEST PRICE HILL,WEST PRICE HILL,01/28/2015 02:41:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2015 02:41:00 PM,...,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WEST PRICE HILL,D - DRIVER,1 - CLEAR,06 - SPORT UTILITY VEHICLE,0,0
258670,39.136956,-84.605497,26-30,WESTWOOD,WESTWOOD,05/31/2014 12:20:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,05/31/2014 12:26:00 PM,...,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",WESTWOOD,D - DRIVER,1 - CLEAR,04 - FULL SIZE,0,0


In [72]:
df['target'].value_counts()

target
0    191120
1     67552
Name: count, dtype: int64

Прекрасно! __67__,552 строчек аварий с травмами. И 191,120 строчек аварий без травм (только проперти дэмэдж).

Теперь аггрегируем участников аварий, чтобы мы могли детально знать кол-во пострадавших и прочие показатели


### Агрегируем количество участников


In [111]:
num_people = df.groupby('localreportno').size().reset_index(name='num_people')
num_people.head()

,localreportno,num_people
0,14,2
1,15,1
2,30,2
3,31,3
4,1455001,1


In [ ]:
df['num_people'] = g['localreportno'].transform('size')
num_people['num_people'].describe()

0         2
1         2
2         5
3         2
4         2
         ..
258667    2
258668    2
258669    2
258670    2
258671    2
Name: num_people, Length: 258672, dtype: int64

In [76]:
df['num_drivers'] = g['typeofperson'].transform(lambda x: (x == 'D - DRIVER').sum())
df['num_occupants'] = g['typeofperson'].transform(lambda x: (x == 'O - OCCUPANT').sum())
df['num_pedestrians'] = g['typeofperson'].transform(lambda x: (x == 'P - PEDESTRIAN').sum())
df[['num_people', 'num_drivers', 'num_occupants', 'num_pedestrians']]

,num_people,num_drivers,num_occupants,num_pedestrians
0,2,2,0,0
1,2,1,1,0
2,5,2,3,0
3,2,2,0,0
4,2,2,0,0
...,...,...,...,...
258667,2,2,0,0
258668,2,2,0,0
258669,2,2,0,0
258670,2,2,0,0


In [77]:
#сделаем и по флагам тож
df['has_driver'] = g['typeofperson'].transform(lambda x: int((x == 'D - DRIVER').any()))
df['has_occupant'] = g['typeofperson'].transform(lambda x: int((x == 'O - OCCUPANT').any()))
df['has_pedestrian'] = g['typeofperson'].transform(lambda x: int((x == 'P - PEDESTRIAN').any()))
df[['has_driver', 'has_occupant', 'has_pedestrian']]

,has_driver,has_occupant,has_pedestrian
0,1,0,0
1,1,1,0
2,1,1,0
3,1,0,0
4,1,0,0
...,...,...,...
258667,1,0,0
258668,1,0,0
258669,1,0,0
258670,1,0,0


__лол. в 675 авариях не было водителя. интересно__

In [78]:
df['num_males'] = g['gender'].transform(lambda x: (x == 'M - MALE').sum())
df['num_females'] = g['gender'].transform(lambda x: (x == 'F - FEMALE').sum())
df[['num_males', 'num_females']]

,num_males,num_females
0,1,1
1,1,1
2,4,1
3,1,1
4,2,0
...,...,...
258667,1,1
258668,2,0
258669,1,1
258670,2,0


In [79]:
df['has_bus'] = g['unittype'].transform(lambda x: int(x.astype(str).str.contains('BUS', case=False, na=False).any()))
df['has_motorcycle'] = g['unittype'].transform(lambda x: int(x.astype(str).str.contains('MOTORCYCLE', case=False, na=False).any()))
df['has_bicycle'] = g['unittype'].transform(lambda x: int(x.astype(str).str.contains('BICYCLE', case=False, na=False).any()))

df[['has_bus', 'has_motorcycle', 'has_bicycle']]

,has_bus,has_motorcycle,has_bicycle
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
...,...,...,...
258667,0,0,0
258668,0,0,0
258669,0,0,0
258670,0,0,0


In [87]:
px.histogram(df, x='target', title='распределение аварий без постральных и с пострадавшими')


### Пощупаем гипотезы и поищем эвристики

In [89]:
df.groupby('has_pedestrian')['target'].mean()

has_pedestrian
0    0.245315
1    0.948815
Name: target, dtype: float64

<mark>То есть если в дтп участвовал пешеход, то почти в 95% случаев это дтп не относится к категории "property damage only. Лучше сидеть дома и никуда не выходить(</mark>

In [92]:
df.groupby('mannerofcrash')['target'].agg(['count', 'mean']).sort_values('mean', ascending=False)

,count,mean
mannerofcrash,,
3 - HEAD-ON,5455,0.580385
1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES IN TRANSPORT,38164,0.347029
6 - ANGLE,76855,0.320148
2 - REAR-END,76715,0.259089
"8 - SIDESWIPE, OPPOSITE DIRECTION",5817,0.221936
4 - REAR-TO-REAR,1018,0.211198
"7 - SIDESWIPE, SAME DIRECTION",41688,0.104419
5 - BACKING,8376,0.065783
9 - UNKNOWN,4583,0.054768


In [100]:
px.bar(df.groupby('mannerofcrash')['target'].agg(['count', 'mean']).reset_index().sort_values('mean', ascending=False), x='mannerofcrash', y='mean', title='доля аварий с пострадавшими в зависимости от типа столкновения')

Наибольшая доля дтп с пострадавшими по типу столкновения наблюдается при лобовом столкновении ——— ≈58%. 

Также выше среднего риск у случаев, не являющихся столкновением между двумя движущимися транспортами, а также у угловых столкновений. Самые низкие значения у BACKING и SIDESWIPE, SAME DIRECTION.

In [96]:
df.groupby('weather')['target'].agg(['count', 'mean']).sort_values('mean', ascending=False)

,count,mean
weather,,
7 - SEVERE CROSSWINDS,53,0.377358
"5 - SLEET, HAIL",403,0.277916
"8 - BLOWING SAND, SOIL, DIRT, SNOW",47,0.276596
4 - RAIN,37844,0.272117
2 - CLOUDY,45778,0.271156
"3 - FOG, SMOG, SMOKE",346,0.265896
1 - CLEAR,167411,0.258740
6 - SNOW,4989,0.238926
"5 - SLEET,HAIL",69,0.202899


Гипотеза: неблагоприятные погодные условия повышают вероятность ДТП с пострадавшими.

In [ ]:
px.bar(df.groupby('weather')['target'].agg(['count', 'mean']).sort_values('mean', ascending=False).reset_index(), 
       x='weather', y='mean', title='доля аварий с пострадавшими в зависимости от погодных условий')

Неожиданный инсайт: <mark> погода влияет слабее, чем пешеходы или тип столкновения </mark>

Разница между CLEAR, CLOUDY, RAIN, FOG небольшая. Погодные условия не выглядят очень сильным фактором тяжести ДТП.

In [107]:
df.groupby('lightconditionsprimary')['target'].agg(['count', 'mean']).sort_values('mean', ascending=False)

,count,mean
lightconditionsprimary,,
3 - DARK - LIGHTED ROADWAY,15910,0.323696
9 - OTHER,39,0.307692
2 - DUSK,1745,0.289398
4 - DARK - LIGHTED ROADWAY,45152,0.281604
4 - DARK – ROADWAY NOT LIGHTED,918,0.260349
2 - DAWN,5282,0.254828
1 - DAYLIGHT,179885,0.254346
3 - DUSK,4914,0.246642
5 - DARK – ROADWAY NOT LIGHTIED,1100,0.212727


Гипотеза: недостаточная освещённость повышает вероятность дтп с пострадавшими.

In [109]:
px.bar(df.groupby('lightconditionsprimary')['target'].agg(['count', 'mean']).sort_values('mean', ascending=False).reset_index(), 
       x='lightconditionsprimary', y='mean', title='доля аварий с пострадавшими в зависимости от освещения')

Одна и та же категория встречается с разными числовыми кодами. Пупупу. Видимо это степени


ДТП в тёмное время суток на освещённой дороге имеют более высокую долю пострадавших (28–32%), чем ДТП днём (25%).

In [110]:
df

,latitude_x,longitude_x,age,community_council_neighborhood,cpd_neighborhood,crashdate,crashlocation,crashseverity,crashseverityid,datecrashreported,...,num_occupants,num_pedestrians,has_driver,has_occupant,has_pedestrian,num_males,num_females,has_bus,has_motorcycle,has_bicycle
0,39.107808,-84.688195,31-40,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,06/17/2014 05:29:00 PM,...,0,0,1,0,0,1,1,0,0,0
1,39.108110,-84.560280,18-25,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,02/15/2015 03:10:00 PM,...,1,0,1,1,0,1,1,0,0,0
2,39.135486,-84.496520,18-25,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,07/23/2015 11:54:00 PM,...,3,0,1,1,0,4,1,0,0,0
3,39.147889,-84.489222,61-70,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/21/2018 01:16:00 PM,...,0,0,1,0,0,1,1,0,0,0
4,39.110989,-84.573138,31-40,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,09/01/2018 07:59:00 PM,...,0,0,1,0,0,2,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258667,39.159659,-84.402688,61-70,MADISONVILLE,MADISONVILLE,04/17/2018 04:46:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,04/17/2018 04:51:00 PM,...,0,0,1,0,0,1,1,0,0,0
258668,39.140463,-84.602730,18-25,WESTWOOD,WESTWOOD,11/06/2014 10:21:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,11/06/2014 10:22:00 PM,...,0,0,1,0,0,2,0,0,0,0
258669,39.113840,-84.592351,51-60,WEST PRICE HILL,WEST PRICE HILL,01/28/2015 02:41:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,01/28/2015 02:41:00 PM,...,0,0,1,0,0,1,1,0,0,0
258670,39.136956,-84.605497,26-30,WESTWOOD,WESTWOOD,05/31/2014 12:20:00 PM,02 - FOUR-WAY INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,05/31/2014 12:26:00 PM,...,0,0,1,0,0,2,0,0,0,0
